# Multimodal RecSys Agent — Kaggle Notebook

Run cells top-to-bottom. Each cell is self-contained.

| # | Cell | Time |
|---|------|------|
| 1 | GPU check | < 1s |
| 2 | Clone repo | ~30s |
| 3 | Install deps | ~2 min |
| 4 | Download data | ~5 min |
| 5 | Preprocess | ~3 min |
| 6 | W&B secret | < 1s |
| 7 | Config setup | < 1s |
| 8 | Train Mult-VAE | ~45 min (10 epochs) |
| 9 | Two-Tower sanity check | ~5 min (5 epochs) |
| 10 | Two-Tower full training | ~30 min (30 epochs) |
| 10b | iALS fallback (only if BPR flat) | ~5 min |
| 11 | Build Qdrant index | ~3 min |
| 12 | RecSys eval | ~2 min |
| 13 | Save checkpoints | < 1s |
| 14 | (Optional) Agent with Ollama | ~10 min |


In [ ]:
# Cell 1: GPU Check
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# Cell 2: Clone Repo
import os

REPO_URL = "https://github.com/MadhaviSG/multimodal-recsys-agent.git"
WORK_DIR = "/kaggle/working/multimodal-recsys-agent"

if not os.path.exists(WORK_DIR):
    !git clone {REPO_URL} {WORK_DIR}

os.chdir(WORK_DIR)
os.environ["PYTHONPATH"] = WORK_DIR
print(f"Working directory: {os.getcwd()}")


In [ ]:
# Cell 3: Install Dependencies
!pip install -q qdrant-client rank-bm25 lightgbm diffusers accelerate python-dotenv scipy implicit
!pip install -q FlagEmbedding
!pip install -q wandb
print("Dependencies installed.")


In [ ]:
# Cell 4: Download Data
!python scripts/download_data.py


In [ ]:
# Cell 5: Preprocess
!python scripts/preprocess.py

import scipy.sparse as sp, pandas as pd
train = sp.load_npz("data/processed/train.npz")
print(f"Train matrix: {train.shape}, nnz={train.nnz:,}")
ratings = pd.read_csv("data/raw/ml-25m/ratings.csv")
print(f"Ratings: {len(ratings):,} rows")


In [ ]:
# Cell 6: W&B Secret
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
print("W&B key loaded.")


In [ ]:
# Cell 7: Pull latest code
!git pull origin main


In [ ]:
# Cell 8: Train Mult-VAE  (~45 min for 10 epochs on T4)
!python scripts/train_mult_vae.py --config configs/config.yaml


In [ ]:
# Cell 9: Two-Tower Architecture Check  (verify model loads correctly)
import torch
from src.recsys.models.two_tower import TwoTowerModel

model = TwoTowerModel(num_users=10, num_items=10, item_feature_dim=21)
print(model)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
# Expected: user_embed (10, 64) + item_embed (10, 64) only — no MLP layers


In [ ]:
# Cell 10: Two-Tower Sanity Check (~5 min)
# Watch first 5 epoch lines. Loss should decrease epoch 1 → 5.
# Ctrl+C after epoch 5 if it's learning — proceed to Cell 11.
# If loss stuck at 0.693 → run Cell 10b (iALS fallback) instead.
!python scripts/train_two_tower.py --config configs/config.yaml


In [ ]:
# Cell 11: Two-Tower Full Training (~30 min)
!python scripts/train_two_tower.py --config configs/config.yaml


In [ ]:
# Cell 10b: iALS Fallback  (~5 min — run ONLY if BPR loss was flat in Cell 10)
# AlternatingLeastSquares on the sparse interaction matrix.
# Outputs: checkpoints/item_embeddings.npy (26224, 64) + two_tower_best.pt
# State-dict keys match TwoTowerModel — rest of pipeline unchanged.

import numpy as np, scipy.sparse as sp, torch, torch.nn.functional as F
from pathlib import Path
import implicit

train_matrix = sp.load_npz("data/processed/train.npz")
n_users, n_items = train_matrix.shape
print(f"Matrix: {n_users:,} users × {n_items:,} items, nnz={train_matrix.nnz:,}")

conf = train_matrix.astype(np.float32)
conf.data[:] = 1.0
item_user = conf.T.tocsr()

als = implicit.als.AlternatingLeastSquares(
    factors=64, iterations=30, regularization=0.01, random_state=42,
)
print("Fitting ALS...")
als.fit(item_user)

item_factors = np.array(als.item_factors)   # (n_items, 64)
user_factors = np.array(als.user_factors)   # (n_users, 64)

item_embs = F.normalize(torch.tensor(item_factors, dtype=torch.float32), dim=-1).numpy()
user_embs = F.normalize(torch.tensor(user_factors, dtype=torch.float32), dim=-1).numpy()

Path("checkpoints").mkdir(exist_ok=True)
np.save("checkpoints/item_embeddings.npy", item_embs)
torch.save({
    "epoch": 30,
    "model_state_dict": {
        "user_embed.weight": torch.tensor(user_factors, dtype=torch.float32),
        "item_embed.weight": torch.tensor(item_factors, dtype=torch.float32),
    },
    "recall": 0.0, "config": {}, "n_users": n_users, "n_items": n_items,
    "num_items": n_items, "item_feature_dim": 21,
}, "checkpoints/two_tower_best.pt")
print(f"item_embeddings.npy: {item_embs.shape}")
print("two_tower_best.pt saved")

val_matrix = sp.load_npz("data/processed/val.npz")
rng = np.random.RandomState(42)
item_t = torch.tensor(item_embs)
user_t = torch.tensor(user_embs)
recalls = []
for uid in rng.choice(n_users, size=200, replace=False):
    scores = (user_t[uid] @ item_t.T).numpy()
    scores[train_matrix[uid].indices] = -np.inf
    val_items = set(val_matrix[uid].indices)
    if not val_items:
        continue
    top10 = np.argsort(scores)[::-1][:10]
    recalls.append(sum(1 for i in top10 if i in val_items) / len(val_items))
print(f"ALS Recall@10 (200 users): {np.mean(recalls):.4f}")


In [ ]:
# Cell 12: Build Qdrant Index  (~3 min)
import yaml

with open("configs/config.yaml") as f:
    config = yaml.safe_load(f)
config["retrieval"]["qdrant_url"] = ":memory:"
with open("configs/config.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

!python scripts/build_index.py --config configs/config.yaml --recreate


In [ ]:
# Cell 13: RecSys Eval — Mult-VAE  (~2 min, 100 users)
import sys, json
import numpy as np, scipy.sparse as sp
import torch
sys.path.insert(0, ".")

from src.eval.recsys.metrics import ndcg_at_k, recall_at_k
from src.recsys.models.mult_vae import MultVAE

train_mat = sp.load_npz("data/processed/train.npz")
val_mat   = sp.load_npz("data/processed/val.npz")
n_users, n_items = train_mat.shape
device = "cuda" if torch.cuda.is_available() else "cpu"

ckpt = torch.load("checkpoints/mult_vae_best.pt", map_location=device)
model = MultVAE(
    num_items=n_items,
    hidden_dims=ckpt["config"]["recsys"]["hidden_dims"],
    latent_dim=ckpt["config"]["recsys"]["latent_dim"],
).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

K = 10
rng = np.random.RandomState(42)
ndcg_scores, recall_scores = [], []

with torch.no_grad():
    for uid in rng.choice(n_users, size=100, replace=False):
        train_row = train_mat[uid].toarray().squeeze(0)
        x = torch.tensor(train_row, dtype=torch.float32).unsqueeze(0).to(device)
        recon, _, _ = model(x)
        scores = recon.squeeze(0).cpu().numpy()
        scores[train_row > 0] = -np.inf

        val_row = val_mat[uid].toarray().squeeze(0)
        relevant = set(np.where(val_row > 0)[0])
        if not relevant:
            continue
        top_k = np.argsort(scores)[::-1][:K]
        ndcg_scores.append(ndcg_at_k(list(top_k), relevant, K))
        recall_scores.append(recall_at_k(list(top_k), relevant, K))

print(f"=== Mult-VAE Eval (100 users) ===")
print(f"NDCG@{K}:   {np.mean(ndcg_scores):.4f}  (paper baseline: ~0.44 at 50 epochs)")
print(f"Recall@{K}: {np.mean(recall_scores):.4f}")


In [ ]:
# Cell 14: Save Checkpoints to Kaggle Output
import os, shutil

shutil.copytree("checkpoints",     "/kaggle/working/checkpoints",    dirs_exist_ok=True)
shutil.copytree("data/processed",  "/kaggle/working/data_processed", dirs_exist_ok=True)

print("Saved to /kaggle/working:")
for fname in sorted(os.listdir("/kaggle/working/checkpoints")):
    size = os.path.getsize(f"/kaggle/working/checkpoints/{fname}") / 1e6
    print(f"  checkpoints/{fname}: {size:.1f} MB")


In [ ]:
# Cell 15 (Optional): Agent with Ollama — no OpenAI key required
import subprocess, time, yaml

!curl -fsSL https://ollama.com/install.sh | sh
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!ollama pull phi3:mini

with open("configs/config.yaml") as f:
    config = yaml.safe_load(f)
config["agent"]["llm_model"] = "phi3:mini"
with open("configs/config.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

!python scripts/run_agent.py --query "Recommend me psychological thrillers"
